# Explorar a camada Gold

Notebook para olhar os dados do lake local (`.data/lake/`) com SQL, via DuckDB.
Cada tabela é um `data.parquet`. A célula de setup registra todas como *views*, então dá para consultar pelo nome.

**Antes de rodar:** gere a Gold com `uv run --locked python -m pipelines.cli.build_gold --local --root .data`.

**Cuidado ao commitar:** limpe as saídas (*Edit → Clear All Outputs*) antes do commit, para não versionar dados.

Para mudar o município analisado, troque `MUNICIPIO` na célula de parâmetros.

## 1. Setup

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# Sobe até a raiz do repositório (onde está o pyproject.toml), de onde quer que o notebook rode
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LAKE = ROOT / ".data" / "lake"

# Do Silver, só as dimensões com nomes de municípios e freguesias. Os anúncios individuais do Idealista
# ficam de fora de propósito: pela constitution, só agregados podem ser publicados.
pastas = [LAKE / "silver" / "dim_municipality", LAKE / "silver" / "dim_parish", *sorted((LAKE / "gold").glob("*/"))]

con = duckdb.connect()
tabelas = []
for pasta in pastas:
    arquivo = pasta / "data.parquet"
    if arquivo.exists():
        con.sql(f"create or replace view {pasta.name} as select * from read_parquet('{arquivo.as_posix()}')")
        tabelas.append((pasta.parent.name, pasta.name))


def q(sql: str) -> pd.DataFrame:
    """Roda SQL no DuckDB e devolve um DataFrame."""
    return con.sql(sql).df()


print(f"{len(tabelas)} views registradas a partir de {LAKE}")

## 2. Parâmetros

In [ ]:
MUNICIPIO = "Lisboa"          # nome oficial (dim_municipality.official_name)
CRITERIO = "custo_aluguel_mediana"   # um criterion_id do catálogo (seção 4)

## 3. Inventário: tabelas e número de linhas

In [ ]:
pd.DataFrame(
    [(camada, nome, con.sql(f"select count(*) from {nome}").fetchall()[0][0]) for camada, nome in tabelas],
    columns=["camada", "tabela", "linhas"],
)

## 4. Catálogo de critérios

O que cada critério significa, a unidade, se *maior* ou *menor* é melhor, o peso e o status.

In [ ]:
q("""
select category, criterion_id, label_pt, unit, direction, weight, status, levels, source_ids
from gold_criteria_catalog
order by category, criterion_id
""")

In [ ]:
# Metodologia e atribuição de um critério
q(f"""
select criterion_id, methodology_pt, attributions, usage_restrictions, reference_periods
from gold_criteria_catalog where criterion_id = '{CRITERIO}'
""").T

## 5. Qualidade e cobertura

Quantos territórios têm valor em cada critério, quantos estão ausentes, zerados por convenção da fonte, herdados do município etc.

In [ ]:
cobertura = q("""
select criterion_id, level, variant, status, territories_expected, with_value, missing,
       zero_by_convention, older_period, inherited, outliers,
       round(100.0 * with_value / nullif(territories_expected, 0), 1) as pct_cobertura
from gold_data_quality
order by level, pct_cobertura, criterion_id
""")
cobertura

In [ ]:
# Só o que está incompleto no nível municipal
cobertura[(cobertura["level"] == "municipio") & (cobertura["pct_cobertura"].fillna(0) < 100)]

## 6. Perfil de um município: todos os indicadores

In [ ]:
dico = q(f"select dico from dim_municipality where official_name = '{MUNICIPIO}'")["dico"].item()

perfil = q(f"""
select i.category, i.criterion_id, c.label_pt, i.variant, i.value, i.unit, i.period,
       i.value_status, i.missing_reason, i.origin, i.origin_level, i.is_older_period, i.source_id
from gold_municipality_indicator i
join gold_criteria_catalog c using (criterion_id)
where i.dico = '{dico}'
order by i.category, i.criterion_id, i.variant
""")
print(MUNICIPIO, dico)
perfil

## 7. Comparar municípios num critério

Ranking pelo valor bruto. A `direction` do catálogo diz se maior ou menor é melhor.

In [ ]:
ranking = q(f"""
select m.official_name as municipio, m.district_or_island as distrito, i.variant, i.value, i.unit, i.period,
       i.origin_level
from gold_municipality_indicator i
join dim_municipality m using (dico)
where i.criterion_id = '{CRITERIO}' and i.value is not null
  and i.variant in ('', 'ALL')
order by i.value desc
""")
print(len(ranking), "municípios com valor")
pd.concat([ranking.head(10), ranking.tail(10)])

In [ ]:
# Distribuição do critério e os limites usados na normalização (P5/P95)
ax = ranking["value"].plot.hist(bins=40, title=CRITERIO, figsize=(9, 4))
norm = q(f"select * from gold_criteria_normalization where criterion_id = '{CRITERIO}' and level = 'municipio'")
for _, linha in norm.iterrows():
    ax.axvline(linha["p5"], linestyle="--", color="gray")
    ax.axvline(linha["p95"], linestyle="--", color="gray")
norm

## 8. Aluguel por tipologia

Medianas só existem com pelo menos 10 anúncios elegíveis (sem outliers). Fora disso, `sample_status` explica o motivo.

Sem amostra própria, o valor publicado do município vem do distrito ou da região: veja `reference_level` na tabela
municipal e `origin_level` no perfil.

In [ ]:
q("""
select sample_status, typology, count(*) as municipios
from gold_rent_by_municipality_typology
group by all order by typology, municipios desc
""")

In [ ]:
q("""
select m.official_name as municipio, r.typology, r.n_eligible, r.rent_median, r.rent_p25, r.rent_p75,
       r.eur_m2_median, r.snapshot_month
from gold_rent_by_municipality_typology r
join dim_municipality m using (dico)
where r.rent_median is not null
order by m.official_name, r.typology
""")

### Aluguel T2 e T3 por localidade: região › distrito › município › freguesia

A Gold publica a **mediana**, não a média, porque a média é puxada por poucos anúncios muito caros. Muitos distritos
(Braga, Aveiro, Coimbra, Setúbal, Alentejo, ilhas…) foram extraídos do Idealista só no nível do **distrito**, sem
município. Desde a spec 012, a Gold calcula o aluguel por distrito e por região e o município sem amostra **herda** o
valor da área, marcado.

| Nível | Tabela |
| --- | --- |
| freguesia, município | `gold_rent_by_parish_typology`, `gold_rent_by_municipality_typology` |
| distrito (ou ilha) | `gold_rent_by_district_typology` |
| região | `gold_rent_by_region_typology` |
| valor publicado e sua origem | `gold_municipality_indicator`, `gold_parish_indicator` (`origin_level`) |

A **região** é a do Idealista (Norte, Centro, Área Metropolitana de Lisboa, Algarve, Alentejo, Madeira, Açores), não a
NUTS II oficial. Na fonte, Santarém fica no Alentejo e Setúbal na Área Metropolitana de Lisboa.

Um valor com nível `distrito` ou `regiao` é a referência **da área**, não o aluguel do município.

In [ ]:
q("""
select nivel, territorio,
       max(rent_median) filter (where typology = 'T2') as t2_mediana,
       max(n_eligible)  filter (where typology = 'T2') as t2_anuncios,
       max(rent_median) filter (where typology = 'T3') as t3_mediana,
       max(n_eligible)  filter (where typology = 'T3') as t3_anuncios
from (
    select 'distrito' as nivel, district_or_island as territorio, typology, rent_median, n_eligible
    from gold_rent_by_district_typology
    union all
    select 'regiao', source_region, typology, rent_median, n_eligible
    from gold_rent_by_region_typology
)
group by all
order by nivel desc, t2_mediana desc nulls last
""")

#### Por município

Uma linha por município, com a mediana de T2/T3 de cada nível. As colunas `t2_referencia` e `t3_referencia` são o
valor **publicado** pela Gold (nível mais fino com dado), e `*_nivel` diz de onde ele veio.

In [ ]:
# Valor publicado e nível de origem, direto do perfil (município ou freguesia)
REFERENCIA = """
  max(value)        filter (where variant = 'T2') as t2_referencia,
  max(origin_level) filter (where variant = 'T2') as t2_nivel,
  max(value)        filter (where variant = 'T3') as t3_referencia,
  max(origin_level) filter (where variant = 'T3') as t3_nivel
"""
MEDIANAS = """
  max(rent_median) filter (where typology = 'T2') as t2,
  max(rent_median) filter (where typology = 'T3') as t3
"""

por_municipio = q(f"""
with ref as (select dico, {REFERENCIA} from gold_municipality_indicator
             where criterion_id = 'custo_aluguel_mediana' group by dico),
     mu as (select dico, district_or_island as distrito, source_region as regiao, {MEDIANAS}
            from gold_rent_by_municipality_typology group by all),
     di as (select district_or_island as distrito, {MEDIANAS} from gold_rent_by_district_typology group by all),
     re as (select source_region as regiao, {MEDIANAS} from gold_rent_by_region_typology group by all)
select mu.regiao, mu.distrito, m.official_name as municipio, mu.dico,
       mu.t2 as t2_municipio, di.t2 as t2_distrito, re.t2 as t2_regiao, ref.t2_referencia, ref.t2_nivel,
       mu.t3 as t3_municipio, di.t3 as t3_distrito, re.t3 as t3_regiao, ref.t3_referencia, ref.t3_nivel
from mu
join dim_municipality m using (dico)
join ref using (dico)
left join di using (distrito)
left join re using (regiao)
order by mu.regiao, mu.distrito, municipio
""")
print(len(por_municipio), "municípios;", por_municipio["t2_referencia"].notna().sum(), "com referência de T2")
por_municipio

#### Por freguesia

O mesmo na freguesia: o valor publicado vem da própria freguesia, do município, do distrito ou da região.

In [ ]:
por_freguesia = q(f"""
with ref as (select dicofre, {REFERENCIA} from gold_parish_indicator
             where criterion_id = 'custo_aluguel_mediana' group by dicofre),
     onde as (select distinct dico, district_or_island as distrito, source_region as regiao
              from gold_rent_by_municipality_typology)
select onde.regiao, onde.distrito, m.official_name as municipio, p.official_name as freguesia, p.dicofre,
       ref.t2_referencia, ref.t2_nivel, ref.t3_referencia, ref.t3_nivel
from dim_parish p
join dim_municipality m using (dico)
join onde using (dico)
join ref using (dicofre)
order by onde.regiao, onde.distrito, municipio, freguesia
""")
print(len(por_freguesia), "freguesias")
por_freguesia["t2_nivel"].value_counts(dropna=False)

In [ ]:
# Filtrar por localidade: troque os valores (None = sem filtro)
FILTRO_REGIAO, FILTRO_DISTRITO, FILTRO_MUNICIPIO = None, "Braga", None

filtro = pd.Series(True, index=por_freguesia.index)
for coluna, valor in (("regiao", FILTRO_REGIAO), ("distrito", FILTRO_DISTRITO), ("municipio", FILTRO_MUNICIPIO)):
    if valor is not None:
        filtro &= por_freguesia[coluna] == valor
por_freguesia[filtro]

## 9. Freguesias de um município

`origin` diz se o valor é próprio da freguesia, herdado ou ausente; `origin_level` diz de qual nível veio
(`freguesia`, `municipio`, `distrito` ou `regiao`).

In [ ]:
q(f"""
select origin, origin_level, count(*) as linhas
from gold_parish_indicator where dico = '{dico}'
group by all order by linhas desc
""")

In [ ]:
q(f"""
select f.official_name as freguesia, p.criterion_id, p.variant, p.value, p.unit, p.origin, p.missing_reason
from gold_parish_indicator p
join dim_parish f using (dicofre)
where p.dico = '{dico}' and p.origin = 'proprio'
order by freguesia, p.criterion_id, p.variant
""")

## 10. Clima mensal do município

In [ ]:
clima = q(f"""
select year_month, temp_mean, temp_max_mean, temp_min_mean, precip_total, sun_hours_mean,
       extreme_days, tci, valid_days
from gold_municipality_climate_monthly where dico = '{dico}'
order by year_month
""")
clima.plot(x="year_month", y=["temp_max_mean", "temp_mean", "temp_min_mean"], figsize=(11, 4),
           title=f"Temperatura mensal - {MUNICIPIO}", rot=45)
clima

## 11. Consulta livre

In [ ]:
q("""
select *
from gold_municipality_indicator
limit 20
""")